# Demand forecast

Synthetic daily demand series generated below with a fixed seed, so the notebook runs offline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate

In [ ]:
rng = np.random.default_rng(7)
n = 1460
t = np.arange(n)
demand = (200 + 0.15 * t + 25 * np.sin(2 * np.pi * t / 365.25)
          + 8 * np.sin(2 * np.pi * t / 7) + np.cumsum(rng.normal(0, 3, n)))
dates = pd.date_range("2021-01-01", periods=n, freq="D")
df = pd.DataFrame({"date": dates, "demand": demand})

In [ ]:
df["lag_1"] = df["demand"].shift(1)
df["lag_7"] = df["demand"].shift(7)
df["rolling_7"] = df["demand"].shift(1).rolling(7).mean()
df["day_of_week"] = df["date"].dt.dayofweek
df = df.dropna().reset_index(drop=True)
features = ["lag_1", "lag_7", "rolling_7", "day_of_week"]
X = df[features]
y = df["demand"]
model = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
scoring = ("r2", "neg_mean_absolute_error")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)
shuffled = cross_validate(model, X, y, cv=kf, scoring=scoring)
print(f"shuffled KFold: R2={shuffled['test_r2'].mean():.3f} MAE={-shuffled['test_neg_mean_absolute_error'].mean():.2f}")